# Practice: Build One Figure That Carries an Argument

**Computational Text Analysis Pack · Module 10 — Visualising Text**

*Technique:* From a defensible finding to a single publishable chart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/10-visualising-and-arguing/10e_practice_one_figure.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## The question

> **You have one figure in a journal article. What does it have to show, and
> what does it have to avoid?**

Module 10 taught the traps. This is the constructive version: take a finding
from module 4's practice notebook and build the chart that carries it.

### How to use this notebook

A worked case, not a tutorial. Run it once for the shape of the argument, then
change the question at the top to one of your own.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
import numpy as np
# The finding: which words distinguish two categories, with effect size.
A, B = "health", "politics"          # <- change these
ca, cb = Counter(), Counter()
for t in df[df["category"] == A]["text"]: ca.update(tokenise(str(t)))
for t in df[df["category"] == B]["text"]: cb.update(tokenise(str(t)))
na, nb = sum(ca.values()), sum(cb.values())
rows = [{"word": w, "a": ca[w], "b": cb[w],
         "log_ratio": np.log2(((ca[w]+.5)/na)/((cb[w]+.5)/nb)),
         "total": ca[w]+cb[w]}
        for w in set(ca) | set(cb) if ca[w]+cb[w] >= 40]
k = pd.DataFrame(rows)
print(f"{len(k)} words above the frequency floor")
print(k.nlargest(6, 'log_ratio')[['word','a','b','log_ratio']].to_string(index=False))

## Version 1: the chart most people make

A bar chart of the top 15 words by log-ratio. It is not wrong, and it wastes the
figure: it shows a ranking a table would show better, and hides how many words
there are and how the effect relates to frequency.

In [ ]:
top = k.nlargest(15, "log_ratio")
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.barh(top["word"][::-1], top["log_ratio"][::-1], color="steelblue")
ax.set_xlabel(f"log ratio ({A} vs {B})")
ax.set_title("Version 1: a ranking, which a table does better")
plt.tight_layout()

## Version 2: the chart that carries the argument

Plot every word — frequency against effect size — and label only the extremes.
Now the figure shows the *distribution*: how many words are undistinguished, how
effect size behaves with frequency, and where the words you want to discuss sit
relative to everything else.

This is the shape corpus linguistics uses, and it answers "is this word unusual"
rather than "which word is top".

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.scatter(k["total"], k["log_ratio"], s=9, alpha=.25, color="grey")
ax.set_xscale("log")
ax.axhline(0, color="black", lw=.8)
ax.axhline(1, ls="--", lw=.7, color="crimson")
ax.axhline(-1, ls="--", lw=.7, color="crimson")

for _, r in pd.concat([k.nlargest(8, "log_ratio"), k.nsmallest(8, "log_ratio")]).iterrows():
    ax.annotate(r["word"], (r["total"], r["log_ratio"]), fontsize=8,
                xytext=(3, 2), textcoords="offset points")
ax.set_xlabel("total frequency (log scale)")
ax.set_ylabel(f"log ratio    <- {B}    |    {A} ->")
ax.set_title(f"Vocabulary of {A} against {B}: every word, extremes labelled")
plt.tight_layout()

print(f"words above +1 log-ratio: {(k['log_ratio'] >= 1).sum()}")
print(f"words below -1          : {(k['log_ratio'] <= -1).sum()}")
print(f"words in between        : {((k['log_ratio'].abs() < 1)).sum()}")
print()
print("That last number is the one version 1 concealed. Most vocabulary does not")
print("distinguish the categories at all, and a reader should see that.")

## The checklist for the figure you publish

1. **Does it show the distribution, or only the winners?** Selecting the top n
   before plotting hides the denominator.
2. **Is the axis honest?** A bar chart not starting at zero exaggerates; a log
   axis needs saying so in the caption.
3. **Are the labels the argument?** Label what you will discuss, not everything.
4. **Would it survive being redrawn from your data?** If the figure depends on a
   threshold you chose, the threshold belongs in the caption.
5. **Can a colour-blind reader read it?** Do not encode a distinction in red vs
   green alone.

### 🔧 Your turn 1

Rebuild version 2 for two categories you expect to be *similar*. What does the figure look like when there is no finding?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

The cloud collapses toward the zero line with few labelled extremes — and that
is a publishable figure too, because "these two categories are not lexically
distinguishable in this corpus" is a real result. Being able to show a null
result clearly is a property of this chart type that the top-n bar chart does
not have: a bar chart of the top 15 words always looks like a finding, even when
there isn't one.

</details>